# Landslide Model Validation

Checking data distribution, null values, and validating point extraction logic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import rasterio

## 1. Null Values and Dataset Overview

In [ ]:
df = pd.read_csv('training_data.csv')
print(df.info())
print('\nNull Values:\n', df.isnull().sum())

## 2. Feature Distribution

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
df.drop(columns=['x', 'y', 'label']).hist(ax=axes, bins=30, color='skyblue', edgecolor='black')
plt.tight_layout()
plt.show()

## 3. Coordinate Point Validation
Verify that the extracted feature values at coordinates perfectly match the source rasters, proving predictions are not 'just for show'.

In [ ]:

with open('feature_metadata.json') as f:
    meta = json.load(f)

print(f"Features trained on: {meta['features']}")

sample = df.sample(1).iloc[0]
val_x, val_y = sample['x'], sample['y']
print(f"Testing real sample at X: {val_x}, Y: {val_y}\n")

from rasterio.transform import rowcol
with rasterio.open(meta['raster_files']['slope']) as src:
    row, col = rowcol(src.transform, val_x, val_y)
    actual_slope = src.read(1)[row, col]
    
print(f"Value in training CSV: {sample['slope']}")
print(f"Value directly from TIF: {actual_slope}")
print("MATCH!" if np.isclose(sample['slope'], actual_slope) else "MISMATCH")
